In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta
from dataclasses import dataclass
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("  Backtesting Module Initialized")

## 1. Generate Market Data

In [ ]:
# Generate realistic OHLCV data
np.random.seed(42)
n_days = 1000
dates = pd.date_range(start='2020-01-01', periods=n_days, freq='D')

# Price process with drift and volatility
initial_price = 100
drift = 0.0005  # Daily drift
volatility = 0.02  # Daily volatility

returns = np.random.normal(drift, volatility, n_days)
close_prices = initial_price * np.exp(np.cumsum(returns))

# Generate OHLC
high_prices = close_prices * (1 + np.abs(np.random.normal(0, 0.01, n_days)))
low_prices = close_prices * (1 - np.abs(np.random.normal(0, 0.01, n_days)))
open_prices = np.roll(close_prices, 1)
open_prices[0] = initial_price

# Volume with autocorrelation
base_volume = 1000000
volume = base_volume + np.cumsum(np.random.randn(n_days) * 50000)
volume = np.abs(volume).astype(int)

# Create DataFrame
market_data = pd.DataFrame({
    'date': dates,
    'open': open_prices,
    'high': high_prices,
    'low': low_prices,
    'close': close_prices,
    'volume': volume
})
market_data.set_index('date', inplace=True)

# Calculate additional metrics
market_data['returns'] = market_data['close'].pct_change()
market_data['log_returns'] = np.log(market_data['close'] / market_data['close'].shift(1))
market_data['volatility'] = market_data['returns'].rolling(window=20).std()

# Visualization
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Price chart
axes[0].plot(market_data.index, market_data['close'], 'b-', linewidth=1.5, label='Close')
axes[0].fill_between(market_data.index, market_data['low'], market_data['high'], 
                     alpha=0.2, color='gray', label='High-Low Range')
axes[0].set_ylabel('Price')
axes[0].set_title('Market Price Data')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Returns
axes[1].plot(market_data.index, market_data['returns'], 'g-', linewidth=0.5, alpha=0.7)
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_ylabel('Daily Returns')
axes[1].set_title('Returns Distribution')
axes[1].grid(True, alpha=0.3)

# Volume
axes[2].bar(market_data.index, market_data['volume'], width=1, alpha=0.7, color='orange')
axes[2].set_xlabel('Date')
axes[2].set_ylabel('Volume')
axes[2].set_title('Trading Volume')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Data period: {market_data.index[0]} to {market_data.index[-1]}")
print(f"Total return: {(market_data['close'][-1]/market_data['close'][0] - 1)*100:.2f}%")
print(f"Annualized volatility: {market_data['returns'].std() * np.sqrt(252) * 100:.2f}%")

## 2. Trading Strategy Implementation

In [ ]:
@dataclass
class Trade:
    date: datetime
    action: str  # 'BUY' or 'SELL'
    price: float
    shares: int
    commission: float
    slippage: float
    
    @property
    def total_cost(self):
        return self.price * self.shares + self.commission + self.slippage

class MomentumStrategy:
    """Simple momentum strategy with moving average crossover"""
    
    def __init__(self, fast_period=20, slow_period=50):
        self.fast_period = fast_period
        self.slow_period = slow_period
        
    def generate_signals(self, data: pd.DataFrame) -> pd.Series:
        # Calculate moving averages
        fast_ma = data['close'].rolling(window=self.fast_period).mean()
        slow_ma = data['close'].rolling(window=self.slow_period).mean()
        
        # Generate signals
        signals = pd.Series(0, index=data.index)
        signals[fast_ma > slow_ma] = 1  # Long signal
        signals[fast_ma < slow_ma] = -1  # Short/exit signal
        
        return signals

class MeanReversionStrategy:
    """Bollinger Band mean reversion strategy"""
    
    def __init__(self, period=20, num_std=2):
        self.period = period
        self.num_std = num_std
        
    def generate_signals(self, data: pd.DataFrame) -> pd.Series:
        # Calculate Bollinger Bands
        sma = data['close'].rolling(window=self.period).mean()
        std = data['close'].rolling(window=self.period).std()
        upper_band = sma + self.num_std * std
        lower_band = sma - self.num_std * std
        
        # Generate signals
        signals = pd.Series(0, index=data.index)
        signals[data['close'] < lower_band] = 1  # Oversold - buy
        signals[data['close'] > upper_band] = -1  # Overbought - sell
        
        return signals

# Generate signals for both strategies
momentum_strategy = MomentumStrategy(fast_period=20, slow_period=50)
mean_reversion_strategy = MeanReversionStrategy(period=20, num_std=2)

market_data['momentum_signal'] = momentum_strategy.generate_signals(market_data)
market_data['mean_reversion_signal'] = mean_reversion_strategy.generate_signals(market_data)

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Momentum strategy
axes[0].plot(market_data.index, market_data['close'], 'gray', alpha=0.5, linewidth=1, label='Price')
buy_signals = market_data[market_data['momentum_signal'].diff() == 1]
sell_signals = market_data[market_data['momentum_signal'].diff() == -1]
axes[0].scatter(buy_signals.index, buy_signals['close'], color='green', marker='^', 
               s=100, label='Buy Signal', zorder=5)
axes[0].scatter(sell_signals.index, sell_signals['close'], color='red', marker='v', 
               s=100, label='Sell Signal', zorder=5)
axes[0].set_ylabel('Price')
axes[0].set_title('Momentum Strategy Signals')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mean reversion strategy
axes[1].plot(market_data.index, market_data['close'], 'gray', alpha=0.5, linewidth=1, label='Price')
buy_signals_mr = market_data[market_data['mean_reversion_signal'].diff() == 1]
sell_signals_mr = market_data[market_data['mean_reversion_signal'].diff() == -1]
axes[1].scatter(buy_signals_mr.index, buy_signals_mr['close'], color='green', marker='^', 
               s=100, label='Buy Signal', zorder=5)
axes[1].scatter(sell_signals_mr.index, sell_signals_mr['close'], color='red', marker='v', 
               s=100, label='Sell Signal', zorder=5)
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Price')
axes[1].set_title('Mean Reversion Strategy Signals')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Momentum strategy - Buy signals: {len(buy_signals)}, Sell signals: {len(sell_signals)}")
print(f"Mean reversion strategy - Buy signals: {len(buy_signals_mr)}, Sell signals: {len(sell_signals_mr)}")

## 3. Backtesting Engine with Transaction Costs

In [ ]:
class BacktestEngine:
    """Event-driven backtesting engine with realistic costs"""
    
    def __init__(self, initial_capital=100000, commission_rate=0.001, 
                 slippage_rate=0.0005, position_size=0.95):
        self.initial_capital = initial_capital
        self.commission_rate = commission_rate
        self.slippage_rate = slippage_rate
        self.position_size = position_size
        
        # State variables
        self.cash = initial_capital
        self.position = 0
        self.trades: List[Trade] = []
        self.equity_curve = []
        
    def calculate_commission(self, price: float, shares: int) -> float:
        return price * shares * self.commission_rate
    
    def calculate_slippage(self, price: float, shares: int, action: str) -> float:
        # Slippage is worse for buys (pay more) and sells (receive less)
        slippage_factor = self.slippage_rate if action == 'BUY' else -self.slippage_rate
        return price * shares * abs(slippage_factor)
    
    def execute_trade(self, date: datetime, action: str, price: float, shares: int):
        commission = self.calculate_commission(price, shares)
        slippage = self.calculate_slippage(price, shares, action)
        
        if action == 'BUY':
            total_cost = price * shares + commission + slippage
            if total_cost <= self.cash:
                self.cash -= total_cost
                self.position += shares
                self.trades.append(Trade(date, action, price, shares, commission, slippage))
        elif action == 'SELL':
            if shares <= self.position:
                total_proceeds = price * shares - commission - slippage
                self.cash += total_proceeds
                self.position -= shares
                self.trades.append(Trade(date, action, price, shares, commission, slippage))
    
    def run(self, data: pd.DataFrame, signals: pd.Series) -> pd.DataFrame:
        """Run backtest on data with given signals"""
        self.cash = self.initial_capital
        self.position = 0
        self.trades = []
        self.equity_curve = []
        
        prev_signal = 0
        
        for date, row in data.iterrows():
            signal = signals.loc[date]
            price = row['close']
            
            # Check for signal changes
            if signal != prev_signal:
                if signal == 1 and prev_signal != 1:  # Enter long
                    shares_to_buy = int((self.cash * self.position_size) / price)
                    if shares_to_buy > 0:
                        self.execute_trade(date, 'BUY', price, shares_to_buy)
                elif signal == -1 and self.position > 0:  # Exit long
                    self.execute_trade(date, 'SELL', price, self.position)
            
            # Calculate current equity
            equity = self.cash + self.position * price
            self.equity_curve.append({
                'date': date,
                'equity': equity,
                'cash': self.cash,
                'position': self.position,
                'position_value': self.position * price
            })
            
            prev_signal = signal
        
        return pd.DataFrame(self.equity_curve).set_index('date')

# Run backtests for both strategies
backtest_momentum = BacktestEngine(initial_capital=100000, commission_rate=0.001, slippage_rate=0.0005)
equity_momentum = backtest_momentum.run(market_data, market_data['momentum_signal'])

backtest_mr = BacktestEngine(initial_capital=100000, commission_rate=0.001, slippage_rate=0.0005)
equity_mr = backtest_mr.run(market_data, market_data['mean_reversion_signal'])

# Buy and hold benchmark
buy_hold_shares = int(100000 / market_data['close'].iloc[0])
buy_hold_equity = buy_hold_shares * market_data['close']

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Equity curves
axes[0].plot(equity_momentum.index, equity_momentum['equity'], 'b-', linewidth=2, label='Momentum Strategy')
axes[0].plot(equity_mr.index, equity_mr['equity'], 'g-', linewidth=2, label='Mean Reversion Strategy')
axes[0].plot(market_data.index, buy_hold_equity, 'r--', linewidth=2, alpha=0.7, label='Buy & Hold')
axes[0].axhline(y=100000, color='black', linestyle='--', alpha=0.3, label='Initial Capital')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title('Strategy Performance Comparison')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Returns distribution
momentum_returns = equity_momentum['equity'].pct_change()
mr_returns = equity_mr['equity'].pct_change()
bh_returns = buy_hold_equity.pct_change()

axes[1].hist(momentum_returns.dropna(), bins=50, alpha=0.5, label='Momentum', density=True)
axes[1].hist(mr_returns.dropna(), bins=50, alpha=0.5, label='Mean Reversion', density=True)
axes[1].hist(bh_returns.dropna(), bins=50, alpha=0.5, label='Buy & Hold', density=True)
axes[1].set_xlabel('Daily Returns')
axes[1].set_ylabel('Density')
axes[1].set_title('Returns Distribution')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Momentum Strategy ===")
print(f"Total trades: {len(backtest_momentum.trades)}")
print(f"Final equity: ${equity_momentum['equity'].iloc[-1]:,.2f}")
print(f"Total return: {(equity_momentum['equity'].iloc[-1]/100000 - 1)*100:.2f}%")

print("\n=== Mean Reversion Strategy ===")
print(f"Total trades: {len(backtest_mr.trades)}")
print(f"Final equity: ${equity_mr['equity'].iloc[-1]:,.2f}")
print(f"Total return: {(equity_mr['equity'].iloc[-1]/100000 - 1)*100:.2f}%")

print("\n=== Buy & Hold ===")
print(f"Final equity: ${buy_hold_equity.iloc[-1]:,.2f}")
print(f"Total return: {(buy_hold_equity.iloc[-1]/100000 - 1)*100:.2f}%")

## 4. Performance Metrics

In [ ]:
def calculate_performance_metrics(equity_curve: pd.Series, risk_free_rate=0.02):
    """Calculate comprehensive performance metrics"""
    returns = equity_curve.pct_change().dropna()
    
    # Basic metrics
    total_return = (equity_curve.iloc[-1] / equity_curve.iloc[0]) - 1
    annualized_return = (1 + total_return) ** (252 / len(equity_curve)) - 1
    
    # Risk metrics
    volatility = returns.std() * np.sqrt(252)
    downside_returns = returns[returns < 0]
    downside_volatility = downside_returns.std() * np.sqrt(252)
    
    # Risk-adjusted returns
    sharpe_ratio = (annualized_return - risk_free_rate) / volatility if volatility > 0 else 0
    sortino_ratio = (annualized_return - risk_free_rate) / downside_volatility if downside_volatility > 0 else 0
    
    # Drawdown analysis
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_drawdown = drawdown.min()
    
    # Calmar ratio
    calmar_ratio = annualized_return / abs(max_drawdown) if max_drawdown != 0 else 0
    
    # Win rate
    win_rate = (returns > 0).sum() / len(returns)
    
    # Average win/loss
    avg_win = returns[returns > 0].mean()
    avg_loss = returns[returns < 0].mean()
    profit_factor = abs(avg_win / avg_loss) if avg_loss != 0 else 0
    
    return {
        'Total Return': total_return,
        'Annualized Return': annualized_return,
        'Volatility': volatility,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino_ratio,
        'Max Drawdown': max_drawdown,
        'Calmar Ratio': calmar_ratio,
        'Win Rate': win_rate,
        'Profit Factor': profit_factor
    }

# Calculate metrics for all strategies
metrics_momentum = calculate_performance_metrics(equity_momentum['equity'])
metrics_mr = calculate_performance_metrics(equity_mr['equity'])
metrics_bh = calculate_performance_metrics(buy_hold_equity)

# Create comparison table
metrics_df = pd.DataFrame({
    'Momentum': metrics_momentum,
    'Mean Reversion': metrics_mr,
    'Buy & Hold': metrics_bh
})

print("\n=== Performance Metrics Comparison ===")
print(metrics_df.round(4))

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Sharpe Ratio
strategies = ['Momentum', 'Mean Reversion', 'Buy & Hold']
sharpe_values = [metrics_momentum['Sharpe Ratio'], metrics_mr['Sharpe Ratio'], metrics_bh['Sharpe Ratio']]
axes[0, 0].bar(strategies, sharpe_values, color=['blue', 'green', 'red'], alpha=0.7)
axes[0, 0].set_ylabel('Sharpe Ratio')
axes[0, 0].set_title('Sharpe Ratio Comparison')
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Max Drawdown
drawdown_values = [metrics_momentum['Max Drawdown']*100, metrics_mr['Max Drawdown']*100, 
                   metrics_bh['Max Drawdown']*100]
axes[0, 1].bar(strategies, drawdown_values, color=['blue', 'green', 'red'], alpha=0.7)
axes[0, 1].set_ylabel('Max Drawdown (%)')
axes[0, 1].set_title('Maximum Drawdown Comparison')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Win Rate
win_rate_values = [metrics_momentum['Win Rate']*100, metrics_mr['Win Rate']*100, 
                   metrics_bh['Win Rate']*100]
axes[1, 0].bar(strategies, win_rate_values, color=['blue', 'green', 'red'], alpha=0.7)
axes[1, 0].set_ylabel('Win Rate (%)')
axes[1, 0].set_title('Win Rate Comparison')
axes[1, 0].axhline(y=50, color='black', linestyle='--', alpha=0.3)
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Annualized Return
return_values = [metrics_momentum['Annualized Return']*100, metrics_mr['Annualized Return']*100, 
                 metrics_bh['Annualized Return']*100]
axes[1, 1].bar(strategies, return_values, color=['blue', 'green', 'red'], alpha=0.7)
axes[1, 1].set_ylabel('Annualized Return (%)')
axes[1, 1].set_title('Annualized Return Comparison')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Walk-Forward Analysis

In [ ]:
def walk_forward_analysis(data: pd.DataFrame, strategy, train_size=252, test_size=63):
    """Perform walk-forward analysis"""
    results = []
    
    start_idx = 0
    while start_idx + train_size + test_size < len(data):
        # Split data
        train_data = data.iloc[start_idx:start_idx + train_size]
        test_data = data.iloc[start_idx + train_size:start_idx + train_size + test_size]
        
        # Generate signals on test data
        signals = strategy.generate_signals(test_data)
        
        # Run backtest on test period
        backtest = BacktestEngine(initial_capital=100000)
        equity = backtest.run(test_data, signals)
        
        # Calculate return for this period
        period_return = (equity['equity'].iloc[-1] / equity['equity'].iloc[0]) - 1
        
        results.append({
            'start_date': test_data.index[0],
            'end_date': test_data.index[-1],
            'return': period_return,
            'sharpe': calculate_performance_metrics(equity['equity'])['Sharpe Ratio']
        })
        
        start_idx += test_size
    
    return pd.DataFrame(results)

# Perform walk-forward analysis
wf_momentum = walk_forward_analysis(market_data, momentum_strategy, train_size=252, test_size=63)
wf_mr = walk_forward_analysis(market_data, mean_reversion_strategy, train_size=252, test_size=63)

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Returns over time
axes[0].plot(wf_momentum['start_date'], wf_momentum['return'] * 100, 
            'bo-', linewidth=2, markersize=6, label='Momentum')
axes[0].plot(wf_mr['start_date'], wf_mr['return'] * 100, 
            'go-', linewidth=2, markersize=6, label='Mean Reversion')
axes[0].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Period Return (%)')
axes[0].set_title('Walk-Forward Analysis - Period Returns')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Sharpe ratio over time
axes[1].plot(wf_momentum['start_date'], wf_momentum['sharpe'], 
            'bo-', linewidth=2, markersize=6, label='Momentum')
axes[1].plot(wf_mr['start_date'], wf_mr['sharpe'], 
            'go-', linewidth=2, markersize=6, label='Mean Reversion')
axes[1].axhline(y=0, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Period Start Date')
axes[1].set_ylabel('Sharpe Ratio')
axes[1].set_title('Walk-Forward Analysis - Sharpe Ratio')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n=== Walk-Forward Analysis Summary ===")
print(f"\nMomentum Strategy:")
print(f"  Average period return: {wf_momentum['return'].mean()*100:.2f}%")
print(f"  Win rate: {(wf_momentum['return'] > 0).sum() / len(wf_momentum)*100:.2f}%")
print(f"  Average Sharpe: {wf_momentum['sharpe'].mean():.2f}")

print(f"\nMean Reversion Strategy:")
print(f"  Average period return: {wf_mr['return'].mean()*100:.2f}%")
print(f"  Win rate: {(wf_mr['return'] > 0).sum() / len(wf_mr)*100:.2f}%")
print(f"  Average Sharpe: {wf_mr['sharpe'].mean():.2f}")

## 6. Monte Carlo Simulation

In [ ]:
def monte_carlo_simulation(returns: pd.Series, n_simulations=1000, n_days=252):
    """Run Monte Carlo simulation on strategy returns"""
    # Calculate return statistics
    mean_return = returns.mean()
    std_return = returns.std()
    
    # Run simulations
    simulations = np.zeros((n_simulations, n_days))
    for i in range(n_simulations):
        sim_returns = np.random.normal(mean_return, std_return, n_days)
        simulations[i] = (1 + sim_returns).cumprod()
    
    return simulations

# Run Monte Carlo simulations
momentum_returns = equity_momentum['equity'].pct_change().dropna()
mr_returns = equity_mr['equity'].pct_change().dropna()

mc_momentum = monte_carlo_simulation(momentum_returns, n_simulations=1000, n_days=252)
mc_mr = monte_carlo_simulation(mr_returns, n_simulations=1000, n_days=252)

# Calculate percentiles
momentum_percentiles = np.percentile(mc_momentum, [5, 25, 50, 75, 95], axis=0)
mr_percentiles = np.percentile(mc_mr, [5, 25, 50, 75, 95], axis=0)

# Visualization
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

days = np.arange(252)

# Momentum strategy
for sim in mc_momentum[:100]:  # Plot first 100 simulations
    axes[0].plot(days, sim, 'b-', alpha=0.05, linewidth=0.5)
axes[0].plot(days, momentum_percentiles[2], 'r-', linewidth=3, label='Median')
axes[0].fill_between(days, momentum_percentiles[0], momentum_percentiles[4], 
                     alpha=0.3, color='blue', label='5th-95th Percentile')
axes[0].fill_between(days, momentum_percentiles[1], momentum_percentiles[3], 
                     alpha=0.5, color='blue', label='25th-75th Percentile')
axes[0].axhline(y=1, color='black', linestyle='--', alpha=0.3)
axes[0].set_ylabel('Cumulative Return')
axes[0].set_title('Monte Carlo Simulation - Momentum Strategy (1000 paths)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mean reversion strategy
for sim in mc_mr[:100]:  # Plot first 100 simulations
    axes[1].plot(days, sim, 'g-', alpha=0.05, linewidth=0.5)
axes[1].plot(days, mr_percentiles[2], 'r-', linewidth=3, label='Median')
axes[1].fill_between(days, mr_percentiles[0], mr_percentiles[4], 
                     alpha=0.3, color='green', label='5th-95th Percentile')
axes[1].fill_between(days, mr_percentiles[1], mr_percentiles[3], 
                     alpha=0.5, color='green', label='25th-75th Percentile')
axes[1].axhline(y=1, color='black', linestyle='--', alpha=0.3)
axes[1].set_xlabel('Trading Days')
axes[1].set_ylabel('Cumulative Return')
axes[1].set_title('Monte Carlo Simulation - Mean Reversion Strategy (1000 paths)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Calculate probability of profit
momentum_profit_prob = (mc_momentum[:, -1] > 1).sum() / len(mc_momentum)
mr_profit_prob = (mc_mr[:, -1] > 1).sum() / len(mc_mr)

print("\n=== Monte Carlo Results (1-year horizon) ===")
print(f"\nMomentum Strategy:")
print(f"  Probability of profit: {momentum_profit_prob*100:.2f}%")
print(f"  Expected return (median): {(momentum_percentiles[2, -1] - 1)*100:.2f}%")
print(f"  5th percentile return: {(momentum_percentiles[0, -1] - 1)*100:.2f}%")
print(f"  95th percentile return: {(momentum_percentiles[4, -1] - 1)*100:.2f}%")

print(f"\nMean Reversion Strategy:")
print(f"  Probability of profit: {mr_profit_prob*100:.2f}%")
print(f"  Expected return (median): {(mr_percentiles[2, -1] - 1)*100:.2f}%")
print(f"  5th percentile return: {(mr_percentiles[0, -1] - 1)*100:.2f}%")
print(f"  95th percentile return: {(mr_percentiles[4, -1] - 1)*100:.2f}%")

## 7. Summary

### Key Concepts:

1. **Event-Driven Backtesting**
   - Realistic order execution
   - Position tracking
   - Cash management

2. **Transaction Costs**
   - Commission modeling
   - Slippage estimation
   - Market impact

3. **Performance Metrics**
   - Risk-adjusted returns (Sharpe, Sortino, Calmar)
   - Drawdown analysis
   - Win rates and profit factors

4. **Walk-Forward Analysis**
   - Out-of-sample validation
   - Strategy robustness
   - Temporal stability

5. **Monte Carlo Simulation**
   - Risk assessment
   - Confidence intervals
   - Probability distributions

### Best Practices:
- Always include realistic transaction costs
- Test strategies out-of-sample
- Consider multiple performance metrics
- Account for survivorship bias
- Validate with Monte Carlo simulations
- Monitor strategy degradation over time